# Nori (Synthefy) — Regression

Demonstrates **Nori** on the shared retail/CPG regression datasets. Nori is a
self-hosted tabular foundation model that uses labeled rows as in-context examples
and predicts in a single forward pass, without task-specific training.

> **License note:** the package and public model weights are Apache-2.0. See [`../README.md`](../README.md).

**Compute:** GPU cluster recommended; CPU is supported for these small tables.

**Prerequisite:** run [`shared/notebooks/00_data_preparation.ipynb`](../../../shared/notebooks/00_data_preparation.ipynb) first.


In [ ]:
%pip install synthefy-nori==0.17.1 scikit-learn pandas matplotlib mlflow --quiet

In [ ]:
dbutils.library.restartPython()

## Configuration


In [ ]:
import os, sys

# Make the repo-level common/ package importable.
# Adjust REPO_ROOT if your repo is checked out at a different workspace path.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
COMMON_PATH = os.path.join(REPO_ROOT, "common")
if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from config import CATALOG

# Configure catalog and schema (must match shared/notebooks/00_data_preparation).
SCHEMA = "default"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# MLflow experiment (shared naming convention across vendors).
current_user = spark.sql("SELECT current_user()").collect()[0][0]
MLFLOW_EXPERIMENT_NAME = f"/Users/{current_user}/tabular-fm-databricks"

print(f"Catalog/schema: {CATALOG}.{SCHEMA}")

## Load the Nori regression model

The base `nori` checkpoint is approximately 6M parameters. Change the selector to
`nori-30m` to use the larger model with the same API. The checkpoint downloads and
is cached on the first prediction; no Hugging Face token is required.


In [ ]:
import numpy as np
import torch
import mlflow

from synthefy_nori import NoriRegressor
from evaluation import (
    split_xy, regression_metrics, train_baselines_regression,
    log_result, RESULTS_TABLE,
)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("No GPU detected. Nori supports CPU inference, but a GPU is recommended "
          "for larger tables.")

# text_columns=[] activates Nori's numeric + categorical DataFrame preprocessing
# without loading a text-embedding model. Reuse one estimator so its checkpoint is
# loaded only once across both benchmark tasks.
nori_regressor = NoriRegressor(
    model="nori",
    device=device,
    text_columns=[],
)
print(f"Nori configured on {device}; weights load lazily on first prediction.")

## Helper: evaluate one regression task

In addition to the shared MAE, RMSE, and R2 metrics, this logs empirical coverage
of Nori's native 10th-to-90th-percentile prediction interval to MLflow.


In [ ]:
def evaluate_regression(table_name, target, task_name, test_size=0.2):
    df = spark.table(table_name).toPandas()
    X_train, X_test, y_train, y_test = split_xy(df, target=target, test_size=test_size)
    n_train, n_features, n_test = len(X_train), X_train.shape[1], len(X_test)

    with mlflow.start_run(run_name=f"{task_name}_nori"):
        mlflow.log_params({
            "vendor": "nori", "model_type": "NoriRegressor",
            "model_variant": "nori", "device": device,
            "task": task_name, "problem_type": "regression",
            "n_features": n_features, "train_samples": n_train, "test_samples": n_test,
        })
        nori_regressor.fit(X_train, y_train)
        y_pred = nori_regressor.predict(X_test)
        q10, q90 = nori_regressor.predict(
            X_test, output_type="quantiles", quantiles=[0.1, 0.9]
        )
        y_test_array = np.asarray(y_test)
        interval_coverage = float(np.mean((y_test_array >= q10) & (y_test_array <= q90)))

        metrics = regression_metrics(y_test, y_pred)
        mlflow.log_metrics({k: v for k, v in metrics.items() if v is not None})
        mlflow.log_metric("interval_80_coverage", interval_coverage)
        log_result(spark, vendor="nori", task=task_name, problem_type="regression",
                   model_name="NoriRegressor", metrics=metrics,
                   n_train=n_train, n_test=n_test, n_features=n_features)

    print(f"[{task_name}] Nori: rmse={metrics['rmse']:.4f} "
          f"mae={metrics['mae']:.4f} r2={metrics['r2']:.4f} "
          f"80% interval coverage={interval_coverage:.1%}")

    for name, m in train_baselines_regression(X_train, y_train, X_test, y_test).items():
        log_result(spark, vendor="baseline", task=task_name, problem_type="regression",
                   model_name=name, metrics=m,
                   n_train=n_train, n_test=n_test, n_features=n_features)
        print(f"[{task_name}] {name}: rmse={m['rmse']:.4f} r2={m['r2']:.4f}")
    return metrics, interval_coverage

## Price Elasticity


In [ ]:
price_metrics, price_coverage = evaluate_regression(
    table_name="price_elasticity_train",
    target="price_elasticity",
    task_name="price_elasticity",
)

## Supplier Lead Time


In [ ]:
lead_time_metrics, lead_time_coverage = evaluate_regression(
    table_name="supplier_lead_time_train",
    target="actual_lead_time_days",
    task_name="supplier_lead_time",
)

## Results


In [ ]:
display(
    spark.table(RESULTS_TABLE)
         .where("problem_type = 'regression'")
         .orderBy("task", "vendor", "model_name")
)